In [1]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("RealEstate_Prices.csv")

print("Original Dataset:")
print(df.head())

print("\nDataset Shape:")
print(df.shape)

# Clean column names
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace(r"[^\w]", "", regex=True)
)

print("\nCleaned Column Names:")
print(df.columns.tolist())

Original Dataset:
   Property ID Property Type Neighborhood  Bedrooms  Bathrooms  Area (sq ft)  \
0         1001         Villa     Hillview         3        2.0          1743   
1         1002     Apartment     Hillview         4        4.0          1592   
2         1003     Apartment    Riverside         1        3.0          3618   
3         1004     Apartment       Suburb         4        3.0          1508   
4         1005         Villa     Hillview         3        2.0          2641   

   Year Built  Sale Price ($)  Sale Year  
0        2018          586261       2022  
1        2011          455540       2023  
2        2003          292714       2021  
3        2002          831899       2023  
4        2011          383876       2023  

Dataset Shape:
(100, 9)

Cleaned Column Names:
['property_id', 'property_type', 'neighborhood', 'bedrooms', 'bathrooms', 'area_sq_ft', 'year_built', 'sale_price_', 'sale_year']


In [3]:
print("\nMissing Values Before Handling:")
print(df.isnull().sum())

# Fill numerical missing values with median
df["bathrooms"] = df["bathrooms"].fillna(df["bathrooms"].median())

# Fill categorical missing values with mode
df["neighborhood"] = df["neighborhood"].fillna(
    df["neighborhood"].mode()[0]
)

print("\nMissing Values After Handling:")
print(df.isnull().sum())


Missing Values Before Handling:
property_id      0
property_type    0
neighborhood     2
bedrooms         0
bathrooms        3
area_sq_ft       0
year_built       0
sale_price_      0
sale_year        0
dtype: int64

Missing Values After Handling:
property_id      0
property_type    0
neighborhood     0
bedrooms         0
bathrooms        0
area_sq_ft       0
year_built       0
sale_price_      0
sale_year        0
dtype: int64


In [4]:
# ------------------------------------------------
# 3. DATA MERGING
# ------------------------------------------------

# Example additional neighborhood dataset
neighborhood_data = pd.DataFrame({
    "neighborhood": [
        "Downtown",
        "Suburb",
        "Riverside",
        "Hillview",
        "Lakeside"
    ],
    "avg_income": [
        75000,
        65000,
        60000,
        70000,
        72000
    ]
})

# Merge the datasets
df = pd.merge(
    df,
    neighborhood_data,
    on="neighborhood",
    how="left"
)

print("\nDataset After Merging:")
print(df.head(10))


Dataset After Merging:
   property_id property_type neighborhood  bedrooms  bathrooms  area_sq_ft  \
0         1001         Villa     Hillview         3        2.0        1743   
1         1002     Apartment     Hillview         4        4.0        1592   
2         1003     Apartment    Riverside         1        3.0        3618   
3         1004     Apartment       Suburb         4        3.0        1508   
4         1005         Villa     Hillview         3        2.0        2641   
5         1006     Apartment     Downtown         1        2.0        2273   
6         1007     Townhouse     Downtown         4        1.0        1457   
7         1008     Apartment     Downtown         4        2.0        3559   
8         1009     Townhouse     Downtown         3        3.0         952   
9         1010     Townhouse    Riverside         1        2.0        1727   

   year_built  sale_price_  sale_year  avg_income  
0        2018       586261       2022       70000  
1        2011

In [5]:
# ------------------------------------------------
# 4. FILTER AND SUBSET DATA
# ------------------------------------------------

# Properties sold between 2021 and 2023
filtered_data = df[
    (df["sale_year"] >= 2021) &
    (df["sale_year"] <= 2023)
]

print("\nProperties Sold Between 2021 and 2023:")
print(filtered_data.head())

# Select only houses
houses = df[df["property_type"] == "House"]

print("\nHouse Properties:")
print(houses.head())

# Select important columns
subset = df[
    [
        "property_type",
        "neighborhood",
        "bedrooms",
        "area_sq_ft",
        "sale_price_"
    ]
]


print("\nSelected Columns:")
print(subset.head())


Properties Sold Between 2021 and 2023:
   property_id property_type neighborhood  bedrooms  bathrooms  area_sq_ft  \
0         1001         Villa     Hillview         3        2.0        1743   
1         1002     Apartment     Hillview         4        4.0        1592   
2         1003     Apartment    Riverside         1        3.0        3618   
3         1004     Apartment       Suburb         4        3.0        1508   
4         1005         Villa     Hillview         3        2.0        2641   

   year_built  sale_price_  sale_year  avg_income  
0        2018       586261       2022       70000  
1        2011       455540       2023       70000  
2        2003       292714       2021       60000  
3        2002       831899       2023       65000  
4        2011       383876       2023       70000  

House Properties:
    property_id property_type neighborhood  bedrooms  bathrooms  area_sq_ft  \
21         1022         House     Hillview         4        3.0        2570   
23

In [10]:
# ------------------------------------------------
# 5. HANDLE CATEGORICAL VARIABLES
# ------------------------------------------------

# One-hot encoding
df_encoded = pd.get_dummies(
    df,
    columns=["property_type", "neighborhood"],
    dtype=int
)

print("\nDataset After One-Hot Encoding:")
print(df_encoded.head())

print("\nEncoded Columns:")
print(df_encoded.columns.tolist())


Dataset After One-Hot Encoding:
   property_id  bedrooms  bathrooms  area_sq_ft  year_built  sale_price_  \
0         1001         3        2.0        1743        2018       586261   
1         1002         4        4.0        1592        2011       455540   
2         1003         1        3.0        3618        2003       292714   
3         1004         4        3.0        1508        2002       831899   
4         1005         3        2.0        2641        2011       383876   

   sale_year  avg_income  property_type_Apartment  property_type_House  \
0       2022       70000                        0                    0   
1       2023       70000                        1                    0   
2       2021       60000                        1                    0   
3       2023       65000                        1                    0   
4       2023       70000                        0                    0   

   property_type_Townhouse  property_type_Villa  neighborhood_Dow

In [13]:
# ------------------------------------------------
# 6. AGGREGATE DATA
# ------------------------------------------------

# Average sale price by neighborhood
avg_price_neighborhood = (
    df.groupby("neighborhood")["sale_price_"]
    .mean()
    .sort_values(ascending=False)
)

print("\nAverage Sale Price by Neighborhood:")
print(avg_price_neighborhood)

# Average sale price by property type
avg_price_type = (
    df.groupby("property_type")["sale_price_"]
    .mean()
    .sort_values(ascending=False)
)

print("\nAverage Sale Price by Property Type:")
print(avg_price_type)

# Average price per square foot
df["price_per_sq_ft"] = (
    df["sale_price_"] / df["area_sq_ft"]
)

print("\nPrice Per Square Foot:")
print(df[["property_id", "price_per_sq_ft"]].head())


Average Sale Price by Neighborhood:
neighborhood
Downtown     552411.750000
Lakeside     534053.750000
Suburb       499499.285714
Riverside    462908.304348
Hillview     425439.347826
Name: sale_price_, dtype: float64

Average Sale Price by Property Type:
property_type
House        524993.250000
Villa        521297.676471
Apartment    467666.222222
Townhouse    461895.260870
Name: sale_price_, dtype: float64

Price Per Square Foot:
   property_id  price_per_sq_ft
0         1001       336.351692
1         1002       286.143216
2         1003        80.904920
3         1004       551.657162
4         1005       145.352518


In [ ]:
# ------------------------------------------------
# 7. IDENTIFY AND HANDLE OUTLIERS
# ------------------------------------------------

# Function to detect outliers using IQR
def detect_outliers(data, column):

    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = data[
        (data[column] < lower_bound) |
        (data[column] > upper_bound)
    ]

    return outliers, lower_bound, upper_bound

In [16]:
# Detect sale price outliers
price_outliers, lower, upper = detect_outliers(
    df,
    "sale_price_"
)

print("\nSale Price Outliers:")
print(price_outliers[
    ["property_id", "sale_price_"]
])

print("\nSale Price Bounds:")
print("Lower Bound:", lower)
print("Upper Bound:", upper)


# Detect area outliers
area_outliers, lower_area, upper_area = detect_outliers(
    df,
    "area_sq_ft"
)

print("\nArea Outliers:")
print(area_outliers[
    ["property_id", "area_sq_ft"]
])


Sale Price Outliers:
    property_id  sale_price_
80         1081      3500000

Sale Price Bounds:
Lower Bound: -328194.375
Upper Bound: 1247386.625

Area Outliers:
    property_id  area_sq_ft
90         1091       12000


In [19]:
# Remove outliers
df_clean = df[
    (df["sale_price_"] >= lower) &
    (df["sale_price_"] <= upper) &
    (df["area_sq_ft"] >= lower_area) &
    (df["area_sq_ft"] <= upper_area)
]

print("\nDataset Shape Before Removing Outliers:")
print(df.shape)

print("\nDataset Shape After Removing Outliers:")
print(df_clean.shape)



Dataset Shape Before Removing Outliers:
(100, 11)

Dataset Shape After Removing Outliers:
(98, 11)


In [20]:
# ------------------------------------------------
# FINAL DATASET
# ------------------------------------------------

print("\nFinal Cleaned Dataset:")
print(df_clean.head())

print("\nFinal Dataset Information:")
print(df_clean.info())


Final Cleaned Dataset:
   property_id property_type neighborhood  bedrooms  bathrooms  area_sq_ft  \
0         1001         Villa     Hillview         3        2.0        1743   
1         1002     Apartment     Hillview         4        4.0        1592   
2         1003     Apartment    Riverside         1        3.0        3618   
3         1004     Apartment       Suburb         4        3.0        1508   
4         1005         Villa     Hillview         3        2.0        2641   

   year_built  sale_price_  sale_year  avg_income  price_per_sq_ft  
0        2018       586261       2022       70000       336.351692  
1        2011       455540       2023       70000       286.143216  
2        2003       292714       2021       60000        80.904920  
3        2002       831899       2023       65000       551.657162  
4        2011       383876       2023       70000       145.352518  

Final Dataset Information:
<class 'pandas.DataFrame'>
Index: 98 entries, 0 to 99
Data column